In [1]:
#Primera Parte Practica 2


#Importación de librerias necesarias
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline

# Explicación de mi Practica

-Objetivo--> El objetivo de esta práctica es desarrollar un modelo de clasificación binaria que pueda determinar si una persona tiene diabetes o no, basándose en un conjunto de características médicas.

-Para ello llevaremos a cabo:

    - Uso de algoritmos de aprendizaje automatico como: SVM (Support Vector Machines) y MLP (Multi-Layer Perceptron).

    - Separar los datos en conjuntos de entrenamiento y prueba (70% para entrenamiento y 30% para prueba).

    -Interpretar las métricas de rendimiento y la matriz de confusión para entender cómo se comporta el modelo en la clasificación de casos diabéticos y no diabéticos.

In [2]:
'''
Datos
'''
#Definicion de ruta y nombre de archivo
url = "https://raw.githubusercontent.com/HugoSolisHompanera/medical-data-classification-ml/main/data/diabetes_dataset.csv"

# Cargar el conjunto de datos desde el CSV
df = pd.read_csv(url)

#Mostrar la salida del CSV en forma de tabla
display(df)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [3]:
# Eliminación de las filas que contengan valores "N. A."

df=df.dropna()
display(df)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [4]:
# Crear los conjuntos X e y
y = df.Outcome.values.astype(int)  # Etiquetas (0 o 1)
caract_cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
               "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]  # Características

X_all = df[caract_cols].values  # Conjunto de características

#Filas y numero de caracteristicas
print(X_all.shape)

(768, 8)


In [5]:
# Crea una lista de tuplas donde cada tupla contiene un conjunto de características y sus etiquetas
datasets = [(X_all, y)]
dataset_names = ["Data All"]

In [6]:
#Numeros filas (ejemplos)
print(y.shape)

(768,)


In [7]:
#Conjunto de caracteristicas

print(X_all)

[[  6.    148.     72.    ...  33.6     0.627  50.   ]
 [  1.     85.     66.    ...  26.6     0.351  31.   ]
 [  8.    183.     64.    ...  23.3     0.672  32.   ]
 ...
 [  5.    121.     72.    ...  26.2     0.245  30.   ]
 [  1.    126.     60.    ...  30.1     0.349  47.   ]
 [  1.     93.     70.    ...  30.4     0.315  23.   ]]


In [8]:
#Imprimimos todas las etiquetas (todas las clases (0 o 1) de los ejemplos)
print(y)

[1 0 1 0 1 0 1 0 1 1 0 1 0 1 1 1 1 1 0 1 0 0 1 1 1 1 1 0 0 0 0 1 0 0 0 0 0
 1 1 1 0 0 0 1 0 1 0 0 1 0 0 0 0 1 0 0 1 0 0 0 0 1 0 0 1 0 1 0 0 0 1 0 1 0
 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 1 1
 1 0 0 1 1 1 0 0 0 1 0 0 0 1 1 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0
 0 0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 1 1 0 0 0 0 1 1 0 0 0 1 0 1 0 1 0 0 0 0 0
 1 1 1 1 1 0 0 1 1 0 1 0 1 1 1 0 0 0 0 0 0 1 1 0 1 0 0 0 1 1 1 1 0 1 1 1 1
 0 0 0 0 0 1 0 0 1 1 0 0 0 1 1 1 1 0 0 0 1 1 0 1 0 0 0 0 0 0 0 0 1 1 0 0 0
 1 0 1 0 0 1 0 1 0 0 1 1 0 0 0 0 0 1 0 0 0 1 0 0 1 1 0 0 1 0 0 0 1 1 1 0 0
 1 0 1 0 1 1 0 1 0 0 1 0 1 1 0 0 1 0 1 0 0 1 0 1 0 1 1 1 0 0 1 0 1 0 0 0 1
 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1 1 0 1 1 0 0 1 0 0 1 0 0 1
 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 1 1 1 0 0 1 0 0 1 0 0 1 0 1 1 0 1 0 1 0 1
 0 1 1 0 0 0 0 1 1 0 1 0 1 0 0 0 0 1 1 0 1 0 1 0 0 0 0 0 1 0 0 0 0 1 0 0 1
 1 1 0 0 1 0 0 1 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1
 0 0 0 1 1 0 0 0 0 0 0 0 

In [9]:
# Definición del espacio de búsqueda de SVM
C_range = np.logspace(-5, 5, 15)
gamma_range = np.logspace(-5, 5, 15)
param_grid_svm = dict(gamma=gamma_range, C=C_range)
nested_cv = 5

grid_svm = GridSearchCV(SVC(), param_grid=param_grid_svm, cv=nested_cv)

# Mostrar el rango de valores a explorar
print(C_range)
print(gamma_range)

[1.00000000e-05 5.17947468e-05 2.68269580e-04 1.38949549e-03
 7.19685673e-03 3.72759372e-02 1.93069773e-01 1.00000000e+00
 5.17947468e+00 2.68269580e+01 1.38949549e+02 7.19685673e+02
 3.72759372e+03 1.93069773e+04 1.00000000e+05]
[1.00000000e-05 5.17947468e-05 2.68269580e-04 1.38949549e-03
 7.19685673e-03 3.72759372e-02 1.93069773e-01 1.00000000e+00
 5.17947468e+00 2.68269580e+01 1.38949549e+02 7.19685673e+02
 3.72759372e+03 1.93069773e+04 1.00000000e+05]


In [10]:
'''
Definición de la búsqueda de parámetros para el clasificador MLP
'''
alpha_range = np.logspace(-5, -1, 5)
hidden_layer_sizes_range=[(50,),(100,),(200,),(500,),(1000,)]

param_grid_mlp = dict(alpha=alpha_range, hidden_layer_sizes=hidden_layer_sizes_range)


grid_mlp = GridSearchCV(MLPClassifier(max_iter=1000,
                                      early_stopping=True), param_grid=param_grid_mlp, cv=nested_cv)

'''
Lista de clasificadores y sus nombres incluidos en el estudio experimental
'''

cls_names = ["SVM","MLP"]

classifiers = [
    make_pipeline(StandardScaler(), grid_svm),
    make_pipeline(StandardScaler(), grid_mlp)]

In [11]:
# Imports necesarios
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

# Esta función evalúa el rendimiento de un modelo de clasificación para un problema binario (2 clases: 0 y 1)
def evalua(y_test, y_pred):
    # Obtener las clases únicas en las etiquetas verdaderas
    classes = np.unique(y_test)
    n_classes = len(classes)

    # Inicializar la matriz de confusión
    confusion_matrix = np.zeros((n_classes, n_classes), dtype=int)

    # Rellenar la matriz de confusión
    for true, pred in zip(y_test, y_pred):
        confusion_matrix[true, pred] += 1  # No se resta 1 porque las clases son 0 y 1

    # Inicializar variables para TP, FP, FN y TN
    TP = confusion_matrix[1, 1]  # Verdaderos positivos
    FP = confusion_matrix[0, 1]  # Falsos positivos
    FN = confusion_matrix[1, 0]  # Falsos negativos
    TN = confusion_matrix[0, 0]  # Verdaderos negativos

    # Calcular la tasa de acierto (accuracy)
    accuracy = (TP + TN) / (TP + TN + FP + FN)



    # Calcula directamente las métricas como precisión, recall y F1-score basándose
    # en los verdaderos positivos y otros valores derivados de la matriz de confusión.
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

    # Crear un diccionario con las métricas
    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1-score': f1
    }

    return metrics  # Devuelve un diccionario con las métricas calculadas

# Esta función entrena un modelo de aprendizaje automático con un conjunto de entrenamiento,
# realiza predicciones sobre un conjunto de prueba y evalúa el rendimiento del modelo.
def predictions_model(X_train, y_train, X_test, y_test, model):
    print('\t' + str(model)[:20], end=' - ')

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = evalua(y_test, y_pred)

    print('OK')


    # Devolver las etiquetas verdaderas y las predicciones
    return y_test, y_pred

In [12]:
# Crear un DataFrame para la matriz de confusión
def conf_mat_df(cm):

    return pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"])

# Esta función carga las predicciones desde un archivo, evalúa el rendimiento de los modelos
# y devuelve un diccionario con métricas, matrices de confusión e informes de clasificación.
def get_results(filename):
    # Cargar las predicciones desde el archivo especificado
    with open(filename, 'rb') as fp:
        all_preds = pickle.load(fp)

    # Extraer nombres de clases y nombres de conjuntos de datos
    cls_names = all_preds.pop("cls_names")
    dataset_names = all_preds.pop("dataset_names")

    # Crear una lista de pares (dataset, classifier) y ordenarlos
    data_cls_pairs = list(all_preds.keys())
    data_cls_pairs.sort()

    results = {}

    # Crear un DataFrame para almacenar la precisión para cada clase
    acc_df = pd.DataFrame(index=dataset_names, columns=cls_names)

    # Inicializar resultados para precisión por dataset
    for dataset in dataset_names:
        results[(dataset, "acc")] = pd.DataFrame(columns=cls_names)

    # Iterar sobre cada par de dataset y clasificador para calcular métricas
    for dataset_name, cls_name in data_cls_pairs:
        y_true, y_pred = all_preds[(dataset_name, cls_name)]

        # Evaluar el rendimiento del modelo y almacenar métricas en los resultados
        acc = evalua(y_true, y_pred)

        # Llenar el DataFrame de precisión
        acc_df.at[dataset_name, cls_name] = acc['accuracy']

        # Obtener matriz de confusión y convertirla a DataFrame
        cm = confusion_matrix(y_true, y_pred)
        cm_df = conf_mat_df(cm)
        results[(dataset_name, cls_name, "cm")] = cm_df

        # Obtener informe de clasificación y convertirlo a DataFrame
        report = classification_report(y_true, y_pred, output_dict=True)
        report_df = pd.DataFrame(report).transpose()
        results[(dataset_name, cls_name, "report")] = report_df

    results["Acc"] = acc_df  # Almacenar el DataFrame de precisión global

    return results

In [13]:
# Esta función ejecuta experimentos de clasificación en varios conjuntos de datos y modelos,
# guarda las predicciones en un archivo especificado y luego procesa los resultados.
def run_all_save(filename):
    all_preds = {}

    for dataset, dataset_name in zip(datasets, dataset_names):
        print(dataset_name)
        X, y = dataset

        # 70% entrenamiento y 30% prueba.
        X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3)

        for model, cls_name in zip(classifiers, cls_names):
            print(cls_name)
            y_test_actual, preds = predictions_model(X_train, y_train, X_test, y_test, model)
            all_preds[(dataset_name, cls_name)] = (y_test_actual, preds)

    all_preds["cls_names"] = cls_names
    all_preds["dataset_names"] = dataset_names

    # Guarda resultados en un archivo
    with open(filename, 'wb') as fp:
         pickle.dump(all_preds, fp)


filename = 'predicciones_diabetes.obj'

# Ejecuta los experimentos
# (esta función debe ser llamada después de definir datasets y classifiers)
run_all_save(filename)

# Llama a la función get_results para cargar y procesar los resultados
results = get_results(filename)

Data All
SVM
	Pipeline(steps=[('st - OK
MLP
	Pipeline(steps=[('st - OK


In [14]:
# Mostrar resultados
df_total = results["Acc"].astype(float)  # Precisión total
df_conf = results[("Data All", "SVM", "cm")].astype(float)  # Matriz de confusión para SVM
df_report = results[("Data All", "SVM", "report")].astype(float)  # Informe de clasificación para SVM


# Accuracy

Explicacion : Estos porcentajes indican la proporción de predicciones correctas (tanto positivas como negativas) realizadas por cada modelo sobre el total de predicciones. En este caso, los porcentajes son más bajos comparados con la Parte 2 de esta práctica, lo que sugiere que este problema de clasificación binaria es más desafiante.

Análisis de los datos:

    -La diferencia entre SVM y MLP es pequeña (menos de 1%), lo que sugiere que ambos modelos tienen un rendimiento similar en este conjunto de datos

    

In [15]:
# Precisión total
df_total


,SVM,MLP
Data All,0.792208,0.805195


# Matriz de confusión

-Explicación: Esta matriz nos da información valiosa sobre el rendimiento del modelo en la clasificación binaria (0: no diabético, 1: diabético).

-Análisis de los datos:

    -Verdaderos Negativos (TN) = 130: El modelo clasificó correctamente 130 casos como no diabéticos.

    -Falsos Positivos (FP) = 20: El modelo clasificó incorrectamente 20 casos no diabéticos como diabéticos.

    -Falsos Negativos (FN) = 36: El modelo clasificó incorrectamente 36 casos diabéticos como no diabéticos.

    -Verdaderos Positivos (TP) = 45: El modelo clasificó correctamente 45 casos como diabéticos.

-Comentarios acerca de la matriz:

1. El modelo tiene un mejor rendimiento en la identificación de casos no diabéticos, que en la identificación de casos diabéticos .

2. Hay un número considerable de falsos negativos (36), lo que significa que el modelo está fallando en identificar algunos casos de diabetes. Esto es preocupante en un contexto médico, ya que podría llevar a que algunos pacientes con diabetes no reciban el diagnóstico adecuado.

3. El modelo muestra un rendimiento aceptable, pero no óptimo para un problema médico crítico como la detección de diabetes.




In [16]:
#Matriz de confusion
df_conf

,Predicted 0,Predicted 1
Actual 0,139.0,11.0
Actual 1,37.0,44.0


# Informe de clasificación

-Este informe nos da una visión más detallada del rendimiento del modelo para cada clase y en general

-Interpretación de los datos:

    -Clase NO Diabética : El modelo es más efectivo en identificar casos no diabéticos.

    -Clase Diabética : El modelo tiene un rendimiento más bajo en la identificación de casos diabéticos.

    -El acurracy : Indica un rendimiento moderado del modelo en general.

    -Análisis de promedios :

        1. El macro average muestra un rendimiento promedio entre las dos clases sin considerar el desequilibrio de clases.

        2. El weighted average, que considera el número de instancias en cada clase, muestra valores ligeramente mejores, indicando que el modelo se beneficia del mayor número de instancias en la clase mayoritaria.




In [17]:
#Resumen de las metricas de rendimiento de un modelo de clasificacion
df_report.round(4)[["precision","recall","f1-score"]]

,precision,recall,f1-score
0,0.7898,0.9267,0.8528
1,0.8000,0.5432,0.6471
accuracy,0.7922,0.7922,0.7922
macro avg,0.7949,0.7349,0.7499
weighted avg,0.7934,0.7922,0.7806


# INFORME GENERAL SOBRE CONCLUSIONES EXTRAS

    -Problemas de Identificación por Clase:
 La clase 1 (Diabético) tiene más problemas de identificación, lo cual como ya hemos dicho anteriormente es bastante grave en cuestiones médicas.            

    -Evaluación de la Tasa de Acierto:
La tasa de acierto general (accuracy) del 75.76% se considera moderada, no elevada.
Para un problema médico crítico como la detección de diabetes, se esperaría una tasa de acierto más alta, idealmente por encima del 80-85%.

    -Efecto del tamaño del conjunto de entrenamiento a la tasa de acierto:

*Para ello he ejecutado mi programa con (70% entrenamiento y 30% test) y posteriormente lo he ejecutado con (80% entrenamiento y 20% test)*

 1. He observado que para SVM hay una mejora significativa al incrementar el conjunto de entrenamiento (5% de aumento de la tasa con el aumento de 70% a 80% en entrenamiento).

 2. Para MLP hay una ligera mejora (sobre un 1% subiendo de 70% a 80% en entrenamiento)

 Conlusion--> La mejora en la tasa de acierto al aumentar el conjunto de entrenamiento sugiere que, si es posible, podría ser beneficioso utilizar una mayor proporción de datos para el entrenamiento.
Sin embargo, es importante mantener un conjunto de prueba lo suficientemente grande para una evaluación confiable del modelo.


# CONLCUSION GENERAL

En resumen, aunque los resultados son prometedores, hay margen para mejorar la identificación de casos diabéticos y se recomienda seguir explorando técnicas adicionales para aumentar la efectividad del modelo.